# 06 - Aplicar el clasificador sobre una base de datos nueva

## Descripción

Este notebook carga el clasificador entrenado en el notebook 05 y lo aplica sobre
las ventanas de audio de la base de datos de test creada en el notebook 04. El mismo proceso se puede aplicar para una base de datos nueva sobre la que se pueda predecir.

El resultado es un archivo CSV con un *logit* para cada ventana. Estos valores se
comparan con las anotaciones manuales en el notebook 07 para evaluar el desempeño
del clasificador.

Este notebook no entrena el modelo y tampoco selecciona el umbral final de
clasificación. Para entrenamiento ver el notebook 05

## Decisiones metodológicas

1. La predicción se realiza sobre una base de datos de test independiente de los
   datos utilizados para entrenar el clasificador.
3. Se utiliza `EXPORT_LOGIT_THRESHOLD = -np.inf` para exportar todas las ventanas,
   no solamente las que superan un umbral determinado. Esto permite evaluar
   posteriormente diferentes umbrales sin repetir la inferencia. Para realizar una prediccion final sobre una base de datos nueva se debe definir `EXPORT_LOGIT_THRESHOLD` con el valor del umbral deseado.  
4. Los *logits* se conservan como puntuaciones continuas. La clasificación binaria
   y el cálculo de las métricas se realizan en el notebook 07.
5. Se mantiene una corrección de compatibilidad para `get_embeddings_batch`, ya que
   algunas versiones de USearch devuelven los embeddings como una tupla.

## Importar librerías

In [ ]:
# @title Imports

from pathlib import Path
import gc

from IPython.display import display
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

from perch_hoplite.agile import classifier
from perch_hoplite.db import brutalism
from perch_hoplite.db import score_functions
from perch_hoplite.db import sqlite_usearch_impl

## Configuración

Editar únicamente esta celda para indicar la base de datos, el modelo y el archivo
de salida. `TARGET_LABEL` debe coincidir exactamente con una de las clases guardadas
en el clasificador.

Los parámetros de revisión controlan únicamente la búsqueda y visualización de
scores; no modifican el CSV final.

In [ ]:
# @title Configuración — editar esta celda

DB_PATH = Path("/mnt/d/Test_data_campana/test_data/audios/perch1_emb")

MODEL_PATH = Path("/mnt/c/Agile/Modelos/agile_classifier_v8.pt")

OUTPUT_CSV_PATH = Path("/mnt/c/Agile/Modelos/agile_classifier_v8_pred.csv")

# Etiqueta objetivo guardada dentro del clasificador.
TARGET_LABEL = "PRTR1"

# La base de test debe estar libre de las anotaciones utilizadas
# durante Agile Modeling. El ground truth se mantiene en un CSV externo.
REQUIRE_EMPTY_ANNOTATIONS = True

# Parámetros para revisar la distribución de scores.
NUM_TOP_RESULTS = 100
BRUTE_SEARCH_SAMPLE_SIZE = 1_000_000
SCORE_SAMPLE_SIZE = 2_048
RNG_SEED = 42

# Si es True, busca ventanas con logits cercanos a MARGIN_TARGET_SCORE.
MARGIN_SAMPLING = False
MARGIN_TARGET_SCORE = 0.0

# -np.inf exporta una fila para cada ventana y clase seleccionada.
EXPORT_LOGIT_THRESHOLD = -np.inf

## Funciones auxiliares

La primera función normaliza la salida de USearch para que siempre sea un arreglo de
NumPy. La segunda permite cerrar explícitamente la conexión con SQLite y liberar el
índice USearch cuando termina la predicción.

In [ ]:
# @title Funciones de compatibilidad y cierre

def get_embeddings_batch_compatible(self, window_ids):
    '''Devuelve un lote de embeddings como un arreglo de NumPy.'''

    embeddings_batch = self.ui.get(window_ids)

    if isinstance(embeddings_batch, tuple):
        embeddings_batch = np.stack(embeddings_batch)

    if not isinstance(embeddings_batch, np.ndarray):
        raise RuntimeError(
            "Se esperaba np.ndarray o tuple, pero USearch devolvió "
            f"{type(embeddings_batch)}."
        )

    return embeddings_batch


def close_hoplite_object(hoplite_object, commit=True):
    '''Guarda y cierra explícitamente un objeto Hoplite.'''

    if hoplite_object is None:
        return

    try:
        if commit:
            hoplite_object.commit()
        else:
            try:
                hoplite_object.db.rollback()
            except Exception:
                pass

    finally:
        try:
            hoplite_object.db.close()
        except Exception:
            pass

        try:
            hoplite_object.ui = None
        except Exception:
            pass

        gc.collect()

## Cargar y validar la base de datos de test

Antes de hacer la predicción se comprueba que la base exista, que contenga ventanas
y que cada ventana registrada en SQLite tenga su embedding correspondiente en
USearch.

También se verifica que la tabla `annotations` esté vacía cuando
`REQUIRE_EMPTY_ANNOTATIONS = True`.

In [ ]:
# @title Abrir y auditar la base de datos

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"No existe la base de datos: {DB_PATH}"
    )

if not (DB_PATH / "hoplite.sqlite").exists():
    raise FileNotFoundError(
        f"No se encontró hoplite.sqlite dentro de: {DB_PATH}"
    )

# Aplicar la corrección de compatibilidad antes de leer embeddings.
sqlite_usearch_impl.SQLiteUSearchDB.get_embeddings_batch = (
    get_embeddings_batch_compatible
)

db = sqlite_usearch_impl.SQLiteUSearchDB.create(str(DB_PATH))

window_ids = np.asarray(
    db.match_window_ids(),
    dtype=np.int64,
)

if window_ids.size == 0:
    raise RuntimeError(
        "La base de datos no contiene ventanas para predecir."
    )

contains_mask = np.asarray(
    db.ui.contains(window_ids),
    dtype=bool,
)

n_windows = int(window_ids.size)
n_embeddings = int(db.count_embeddings())
n_missing_embeddings = int((~contains_mask).sum())
n_recordings = len(list(db.get_all_recordings()))
n_annotations = int(
    db.db.execute(
        "SELECT COUNT(*) FROM annotations"
    ).fetchone()[0]
)

print("Verificación de la base de datos")
print(f"  Grabaciones: {n_recordings:,}")
print(f"  Ventanas SQLite: {n_windows:,}")
print(f"  Embeddings USearch: {n_embeddings:,}")
print(f"  Ventanas sin embedding: {n_missing_embeddings:,}")
print(f"  Anotaciones en la DB: {n_annotations:,}")

if n_missing_embeddings != 0:
    missing_window_ids = window_ids[~contains_mask]

    raise RuntimeError(
        "La base contiene ventanas sin embedding. "
        f"Primeros IDs: {missing_window_ids[:10].tolist()}"
    )

if n_windows != n_embeddings:
    raise RuntimeError(
        "El número de ventanas de SQLite no coincide con el número "
        "de embeddings de USearch."
    )

if REQUIRE_EMPTY_ANNOTATIONS and n_annotations != 0:
    raise RuntimeError(
        "La base de test todavía contiene anotaciones. "
        "Reconstrúyala o límpiela antes de generar las predicciones."
    )

## Cargar y validar el clasificador

El archivo del clasificador conserva los coeficientes, el sesgo y los nombres de las
clases. La posición de `TARGET_LABEL` se obtiene directamente desde
`custom_classifier.classes`.

También se comprueba que la dimensión de los embeddings de la base sea compatible
con la dimensión esperada por el clasificador.

In [ ]:
# @title Cargar el modelo y localizar la etiqueta objetivo

if not MODEL_PATH.is_file():
    raise FileNotFoundError(
        f"No existe el archivo del clasificador: {MODEL_PATH}"
    )

custom_classifier = classifier.LinearClassifier.load(
    str(MODEL_PATH)
)

classifier_labels = tuple(custom_classifier.classes)
n_outputs = int(custom_classifier.beta.shape[1])

if len(classifier_labels) != n_outputs:
    raise RuntimeError(
        "El número de clases guardadas no coincide con el número "
        "de salidas del clasificador."
    )

if TARGET_LABEL not in classifier_labels:
    raise ValueError(
        f"La etiqueta {TARGET_LABEL!r} no está en el clasificador. "
        f"Clases disponibles: {classifier_labels}"
    )

target_label_idx = classifier_labels.index(TARGET_LABEL)
class_query = custom_classifier.beta[:, target_label_idx]
bias = float(custom_classifier.beta_bias[target_label_idx])

verification_ids = window_ids[: min(10, n_windows)]
verification_embeddings = db.get_embeddings_batch(
    verification_ids
)

embedding_dimension = int(verification_embeddings.shape[1])
classifier_dimension = int(class_query.shape[0])

if embedding_dimension != classifier_dimension:
    raise RuntimeError(
        "La dimensión de los embeddings no coincide con la dimensión "
        "esperada por el clasificador: "
        f"{embedding_dimension} != {classifier_dimension}."
    )

print("Clasificador cargado correctamente")
print(f"  Archivo: {MODEL_PATH.name}")
print(f"  Clases: {classifier_labels}")
print(f"  Etiqueta objetivo: {TARGET_LABEL}")
print(f"  target_label_idx: {target_label_idx}")
print(f"  Dimensión de los embeddings: {embedding_dimension}")

## Revisar la distribución de scores

Esta búsqueda permite comprobar si el clasificador produce una distribución
razonable de *logits* y observar los scores más altos. Es una revisión diagnóstica:
no selecciona el umbral definitivo y no modifica las predicciones exportadas.

Si `MARGIN_SAMPLING = True`, la búsqueda se concentra en ventanas cercanas a
`MARGIN_TARGET_SCORE`. Si es `False`, devuelve los scores más altos.

In [ ]:
# @title Buscar y mostrar los scores del clasificador

margin_target = (
    MARGIN_TARGET_SCORE
    if MARGIN_SAMPLING
    else None
)

search_sample_size = min(
    BRUTE_SEARCH_SAMPLE_SIZE,
    n_windows,
)

score_sample_size = min(
    SCORE_SAMPLE_SIZE,
    n_windows,
)

score_fn = score_functions.get_score_fn(
    "dot",
    bias=bias,
    target_score=margin_target,
)

results = brutalism.threaded_brute_search(
    db,
    class_query,
    NUM_TOP_RESULTS,
    score_fn=score_fn,
    sample_size=search_sample_size,
)

scores = brutalism.get_random_embedding_scores(
    db,
    class_query,
    score_fn=score_functions.get_score_fn(
        "dot",
        bias=bias,
    ),
    sample_size=score_sample_size,
    rng_seed=RNG_SEED,
)

ordered_results = sorted(
    results.search_results,
    key=lambda result: result.sort_score,
    reverse=True,
)

top_scores = pd.DataFrame(
    {
        "window_id": [
            result.window_id
            for result in ordered_results
        ],
        "logit": [
            result.sort_score
            for result in ordered_results
        ],
    }
)

# La transformación sigmoid facilita la lectura, pero no implica
# que los valores estén calibrados como probabilidades.
top_scores["sigmoid_score"] = 1 / (
    1 + np.exp(-np.clip(top_scores["logit"], -700, 700))
)

fig, ax = plt.subplots(figsize=(9, 4.5))

ax.hist(
    scores,
    bins=25,
    density=True,
    alpha=0.55,
    color="#3b82a0",
    label="Muestra aleatoria",
)

ax.scatter(
    top_scores["logit"],
    np.zeros(len(top_scores)),
    marker="|",
    color="#b22222",
    alpha=0.6,
    label="Resultados de la búsqueda",
)

ax.axvline(
    0,
    color="gray",
    linestyle="--",
    linewidth=1,
    label="Logit = 0",
)

ax.set(
    title=f"Distribución de scores para {TARGET_LABEL}",
    xlabel="Logit",
    ylabel="Densidad",
)

ax.legend()
plt.show()

display(top_scores.head(20))

## Generar las predicciones

`write_inference_csv` aplica el clasificador a todas las ventanas indicadas y escribe
las filas cuyo *logit* sea mayor que `EXPORT_LOGIT_THRESHOLD`.

En este workflow el valor es `-np.inf`, por lo que se exportan todas las ventanas.
Esto es intencional: el notebook 07 necesita la distribución completa de scores
para realizar la evaluación.

In [ ]:
# @title Ejecutar la inferencia y exportar el CSV

OUTPUT_CSV_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

labels_to_export = (TARGET_LABEL,)

classifier.write_inference_csv(
    custom_classifier,
    db,
    str(OUTPUT_CSV_PATH),
    EXPORT_LOGIT_THRESHOLD,
    labels=labels_to_export,
    window_ids=window_ids,
)

print()
print("Inferencia completa")
print(f"  Archivo generado: {OUTPUT_CSV_PATH}")

## Validar el archivo exportado

La validación confirma que el CSV contiene las columnas esperadas, únicamente la
etiqueta solicitada y una fila por ventana cuando se utiliza `-np.inf`.

In [ ]:
# @title Revisar el resultado de la inferencia

predictions = pd.read_csv(OUTPUT_CSV_PATH)

required_columns = {
    "idx",
    "project",
    "filename",
    "window_start",
    "window_end",
    "label",
    "logits",
}

missing_columns = required_columns.difference(
    predictions.columns
)

if missing_columns:
    raise ValueError(
        "El CSV de predicciones no contiene todas las columnas "
        f"esperadas: {sorted(missing_columns)}"
    )

exported_labels = set(
    predictions["label"].dropna().astype(str)
)

if exported_labels != {TARGET_LABEL}:
    raise RuntimeError(
        "Las etiquetas del CSV no coinciden con TARGET_LABEL. "
        f"Etiquetas encontradas: {sorted(exported_labels)}"
    )

if np.isneginf(EXPORT_LOGIT_THRESHOLD):
    if len(predictions) != n_windows:
        raise RuntimeError(
            "Con un umbral de -np.inf se esperaba una fila por ventana, "
            f"pero se obtuvieron {len(predictions):,} de {n_windows:,}."
        )

if predictions["idx"].duplicated().any():
    raise RuntimeError(
        "El CSV contiene window_id duplicados para la etiqueta exportada."
    )

prediction_summary = pd.DataFrame(
    {
        "resultado": [
            "Grabaciones procesadas",
            "Ventanas procesadas",
            "Filas exportadas",
            "Logit mínimo",
            "Logit mediano",
            "Logit máximo",
        ],
        "valor": [
            predictions["filename"].nunique(),
            n_windows,
            len(predictions),
            predictions["logits"].min(),
            predictions["logits"].median(),
            predictions["logits"].max(),
        ],
    }
)

display(prediction_summary)
display(predictions.head())

## Cerrar la base de datos

Después de completar y validar la exportación se cierra la conexión con SQLite y se
libera el índice USearch. Esto permite abrir la base desde otro notebook sin dejar
conexiones activas en este kernel.

In [ ]:
# @title Cerrar la conexión Hoplite

close_hoplite_object(
    db,
    commit=True,
)

db = None

print("Base de datos cerrada correctamente.")

## Resultado esperado

- El clasificador se aplica únicamente a la base de datos de test.
- La base se valida antes de iniciar la inferencia.
- `agile_classifier_v8_pred.csv` contiene un *logit* para cada ventana y la
  etiqueta `PRTR1`.
- No se aplica todavía una decisión positiva/negativa definitiva.
- La conexión con la base de datos se cierra al finalizar.

El siguiente paso es ejecutar `07_Workflow_evaluar_modelo.ipynb`, donde las
predicciones se comparan con el *ground truth* y se calculan las métricas de
desempeño.